# Export Models to ONNX

This notebook exports the trained TensorFlow/Keras detector and classifier
models to ONNX format for use in the C# application.

Goals:
- load trained `.keras` models
- inspect TensorFlow model outputs
- export detector and classifier to ONNX
- validate exported ONNX models

In [ ]:
from pathlib import Path
import tensorflow as tf
import tf2onnx
import onnx
import keras_cv

print("tf:", tf.__version__)
print("tf2onnx:", tf2onnx.__version__)
print("onnx:", onnx.__version__)
print("keras_cv:", keras_cv.__version__)

from project_config import (
    DETECTOR_MODEL_PATH,
    CLASSIFIER_MODEL_PATH,
    DETECTOR_ONNX_PATH,
    CLASSIFIER_ONNX_PATH,
    DETECTOR_INPUT_SHAPE,
    CLASSIFIER_INPUT_SHAPE,
    ONNX_OPSET,
)

# =========================================================
# HELPERS
# =========================================================
def inspect_outputs(outputs, prefix="output"):
    if isinstance(outputs, dict):
        print(f"{prefix}: dict")
        for k, v in outputs.items():
            print(f"  - {k}: shape={getattr(v, 'shape', None)}, dtype={getattr(v, 'dtype', None)}")
    elif isinstance(outputs, (list, tuple)):
        print(f"{prefix}: {type(outputs).__name__}")
        for i, v in enumerate(outputs):
            print(f"  - {prefix}_{i}: shape={getattr(v, 'shape', None)}, dtype={getattr(v, 'dtype', None)}")
    else:
        print(f"{prefix}: tensor shape={getattr(outputs, 'shape', None)}, dtype={getattr(outputs, 'dtype', None)}")


def export_detector_to_onnx(model_path: Path, output_path: Path):
    model = tf.keras.models.load_model(model_path, compile=False)

    dummy = tf.random.uniform(DETECTOR_INPUT_SHAPE, dtype=tf.float32)
    raw_outputs = model(dummy, training=False)

    print("\n[Detector] Raw TF output structure:")
    inspect_outputs(raw_outputs, prefix="detector_raw")

    signature = (tf.TensorSpec(DETECTOR_INPUT_SHAPE, tf.float32, name="images"),)

    tf2onnx.convert.from_keras(
        model,
        input_signature=signature,
        opset=ONNX_OPSET,
        output_path=str(output_path),
    )

    onnx_model = onnx.load(str(output_path))
    onnx.checker.check_model(onnx_model)

    print(f"\nExported detector: {output_path}")
    print(f"Inputs: {[i.name for i in onnx_model.graph.input]}")
    print(f"Outputs: {[o.name for o in onnx_model.graph.output]}")


def export_classifier_to_onnx(model_path: Path, output_path: Path):
    model = tf.keras.models.load_model(model_path, compile=False)

    dummy = tf.random.uniform(CLASSIFIER_INPUT_SHAPE, dtype=tf.float32)
    raw_outputs = model(dummy, training=False)

    print("\n[Classifier] Raw TF output structure:")
    inspect_outputs(raw_outputs, prefix="classifier_raw")

    signature = (tf.TensorSpec(CLASSIFIER_INPUT_SHAPE, tf.float32, name="images"),)

    tf2onnx.convert.from_keras(
        model,
        input_signature=signature,
        opset=ONNX_OPSET,
        output_path=str(output_path),
    )

    onnx_model = onnx.load(str(output_path))
    onnx.checker.check_model(onnx_model)

    print(f"\nExported classifier: {output_path}")
    print(f"Inputs: {[i.name for i in onnx_model.graph.input]}")
    print(f"Outputs: {[o.name for o in onnx_model.graph.output]}")


# =========================================================
# EXPORT
# =========================================================
export_detector_to_onnx(DETECTOR_MODEL_PATH, DETECTOR_ONNX_PATH)
export_classifier_to_onnx(CLASSIFIER_MODEL_PATH, CLASSIFIER_ONNX_PATH)

print("\nDONE")